In [ ]:
import subprocess, sys

# Clone or pull latest repo
import os
REPO_URL = "https://github.com/sfh20003/myscrapers-sfh20003.git"  # ← change this
REPO_DIR = "myscrapers-sfh20003"

if os.path.exists(REPO_DIR):
    # Already cloned — just pull latest to get newest Results/
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("Pulled latest changes")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Cloned repo")

# Install dependencies
subprocess.run([sys.executable, "-m", "pip", "install",
                "pandas", "matplotlib", "numpy", "Pillow", "-q"], check=True)
print(Dependencies installed")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import json
import glob
import os
from datetime import datetime

RESULTS_DIR = f"{REPO_DIR}/Results"

# ---- Load all preds files ----
preds_files = sorted(glob.glob(f"{RESULTS_DIR}/*-preds.csv"))
print(f"Found {len(preds_files)} prediction files")

all_metrics = []

for f in preds_files:
    # Extract timestamp from filename e.g. 2026040418-preds.csv
    run_id = os.path.basename(f).replace("-preds.csv", "")
    
    try:
        df = pd.read_csv(f)
        df['price']           = pd.to_numeric(df['price'], errors='coerce')
        df['predicted_price'] = pd.to_numeric(df['predicted_price'], errors='coerce')
        df['abs_error']       = pd.to_numeric(df['abs_error'], errors='coerce')
        
        valid = df[df['price'].notna() & df['predicted_price'].notna()]
        if len(valid) == 0:
            continue
            
        mae  = float(valid['abs_error'].mean())
        rmse = float(np.sqrt(((valid['predicted_price'] - valid['price'])**2).mean()))
        mape = float((valid['abs_error'] / valid['price'].replace(0, np.nan)).mean() * 100)
        bias = float((valid['predicted_price'] - valid['price']).mean())
        
        # Parse run_id to datetime
        try:
            run_dt = datetime.strptime(run_id, "%Y%m%d%H")
        except:
            run_dt = datetime.strptime(run_id, "%Y%m%d")
        
        all_metrics.append({
            "run_id":   run_id,
            "run_dt":   run_dt,
            "mae":      mae,
            "rmse":     rmse,
            "mape":     mape,
            "bias":     bias,
            "n_preds":  len(valid),
        })
        print(f"  {run_id}: MAE=${mae:,.0f} RMSE=${rmse:,.0f} MAPE={mape:.1f}% n={len(valid)}")
        
    except Exception as e:
        print(f"  Skipping {f}: {e}")

metrics_df = pd.DataFrame(all_metrics).sort_values("run_dt")
print(f"\n✅ Loaded {len(metrics_df)} runs total")
metrics_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Model Error Trends Over Time", fontsize=16, fontweight='bold')

# MAE
axes[0,0].plot(metrics_df['run_dt'], metrics_df['mae'],
               marker='o', color='steelblue', linewidth=2)
axes[0,0].set_title('MAE Over Time')
axes[0,0].set_ylabel('Mean Absolute Error ($)')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].grid(True, alpha=0.3)

# RMSE
axes[0,1].plot(metrics_df['run_dt'], metrics_df['rmse'],
               marker='s', color='coral', linewidth=2)
axes[0,1].set_title('RMSE Over Time')
axes[0,1].set_ylabel('Root Mean Squared Error ($)')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].grid(True, alpha=0.3)

# MAPE
axes[1,0].plot(metrics_df['run_dt'], metrics_df['mape'],
               marker='^', color='green', linewidth=2)
axes[1,0].set_title('MAPE Over Time')
axes[1,0].set_ylabel('Mean Absolute % Error')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].grid(True, alpha=0.3)

# Bias
axes[1,1].bar(range(len(metrics_df)), metrics_df['bias'],
              color=['red' if b > 0 else 'green' for b in metrics_df['bias']])
axes[1,1].set_title('Bias Over Time (+ = over-predicting)')
axes[1,1].set_ylabel('Bias ($)')
axes[1,1].set_xticks(range(len(metrics_df)))
axes[1,1].set_xticklabels(metrics_df['run_id'], rotation=45, ha='right')
axes[1,1].axhline(y=0, color='black', linewidth=0.8, linestyle='--')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("error_trends.png", dpi=150, bbox_inches='tight')
plt.show()
print("Error trends plotted")

In [ ]:
imp_files = sorted(glob.glob(f"{RESULTS_DIR}/*-permutation_importance.csv"))
print(f"Found {len(imp_files)} importance files")

all_importance = []

for f in imp_files:
    run_id = os.path.basename(f).replace("-permutation_importance.csv", "")
    try:
        imp = pd.read_csv(f)
        imp['run_id'] = run_id
        try:
            imp['run_dt'] = datetime.strptime(run_id, "%Y%m%d%H")
        except:
            imp['run_dt'] = datetime.strptime(run_id, "%Y%m%d")
        all_importance.append(imp)
    except Exception as e:
        print(f"  Skipping {f}: {e}")

if all_importance:
    imp_all = pd.concat(all_importance, ignore_index=True)
    
    # Get top 5 features from latest run
    latest_run = imp_all['run_id'].max()
    top5 = (imp_all[imp_all['run_id'] == latest_run]
            .nlargest(5, 'importance')['feature']
            .tolist())
    print(f"Top 5 features in latest run: {top5}")
    
    # Plot importance trend for top 5 features
    fig, ax = plt.subplots(figsize=(12, 5))
    
    for feat in top5:
        feat_df = imp_all[imp_all['feature'] == feat].sort_values('run_dt')
        if len(feat_df) > 0:
            ax.plot(feat_df['run_dt'], feat_df['importance'],
                   marker='o', linewidth=2, label=feat)
    
    ax.set_title('Top 5 Feature Importance Trends Over Time')
    ax.set_ylabel('Permutation Importance')
    ax.set_xlabel('Run Date')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("importance_trends.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Importance trends plotted")
else:
    print("No importance files found yet")

In [ ]:
pdp_files = sorted(glob.glob(f"{RESULTS_DIR}/*-pdp_top3.png"))

if pdp_files:
    latest_pdp = pdp_files[-1]
    run_id = os.path.basename(latest_pdp).replace("-pdp_top3.png", "")
    print(f"Showing latest PDP from run: {run_id}")
    
    img = mpimg.imread(latest_pdp)
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Partial Dependence Plots — Run {run_id}', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No PDP files found yet")

In [ ]:
if len(metrics_df) > 0:
    print("=" * 65)
    print(f"{'MODEL PERFORMANCE SUMMARY':^65}")
    print("=" * 65)
    print(f"{'Run ID':<15} {'MAE':>10} {'RMSE':>10} {'MAPE':>8} {'Bias':>10} {'N':>5}")
    print("-" * 65)
    for _, row in metrics_df.iterrows():
        print(f"{row['run_id']:<15} "
              f"${row['mae']:>9,.0f} "
              f"${row['rmse']:>9,.0f} "
              f"{row['mape']:>7.1f}% "
              f"${row['bias']:>+9,.0f} "
              f"{row['n_preds']:>5.0f}")
    print("=" * 65)
    
    best_run = metrics_df.loc[metrics_df['mae'].idxmin()]
    latest_run = metrics_df.iloc[-1]
    
    print(f"\n🏆 Best MAE:  ${best_run['mae']:,.0f} on run {best_run['run_id']}")
    print(f"📅 Latest:   ${latest_run['mae']:,.0f} MAE on run {latest_run['run_id']}")
    print(f"📈 Runs tracked: {len(metrics_df)}")

In [ ]:
all_preds = []

for f in preds_files:
    run_id = os.path.basename(f).replace("-preds.csv", "")
    try:
        df = pd.read_csv(f)
        df['run_id'] = run_id
        all_preds.append(df)
    except:
        pass

if all_preds:
    preds_all = pd.concat(all_preds, ignore_index=True)
    preds_all['price']           = pd.to_numeric(preds_all['price'], errors='coerce')
    preds_all['predicted_price'] = pd.to_numeric(preds_all['predicted_price'], errors='coerce')
    preds_all['year']            = pd.to_numeric(preds_all['year'], errors='coerce')
    preds_all['mileage']         = pd.to_numeric(preds_all['mileage'], errors='coerce')
    preds_all = preds_all[preds_all['price'].notna() & preds_all['predicted_price'].notna()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Actual vs Predicted Price Distribution", fontsize=14, fontweight='bold')

    # Histogram overlay
    axes[0].hist(preds_all['price'], bins=30, alpha=0.6,
                 color='steelblue', label='Actual Price')
    axes[0].hist(preds_all['predicted_price'], bins=30, alpha=0.6,
                 color='coral', label='Predicted Price')
    axes[0].set_xlabel('Price ($)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Price Distribution')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Scatter actual vs predicted
    axes[1].scatter(preds_all['price'], preds_all['predicted_price'],
                    alpha=0.5, edgecolors='k', linewidths=0.3, color='steelblue')
    lims = [min(preds_all['price'].min(), preds_all['predicted_price'].min()),
            max(preds_all['price'].max(), preds_all['predicted_price'].max())]
    axes[1].plot(lims, lims, 'r--', label='Perfect Prediction')
    axes[1].set_xlabel('Actual Price ($)')
    axes[1].set_ylabel('Predicted Price ($)')
    axes[1].set_title('Actual vs Predicted Scatter')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("price_distribution.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Price distribution plotted")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Price Analysis by Vehicle Year", fontsize=14, fontweight='bold')

year_df = preds_all[preds_all['year'].between(1990, 2026)].copy()
year_group = year_df.groupby('year').agg(
    actual_price=('price', 'mean'),
    predicted_price=('predicted_price', 'mean'),
    count=('price', 'count')
).reset_index()

# Line chart - actual vs predicted by year
axes[0].plot(year_group['year'], year_group['actual_price'],
             marker='o', color='steelblue', linewidth=2, label='Actual Avg Price')
axes[0].plot(year_group['year'], year_group['predicted_price'],
             marker='s', color='coral', linewidth=2, linestyle='--', label='Predicted Avg Price')
axes[0].set_xlabel('Vehicle Year')
axes[0].set_ylabel('Average Price ($)')
axes[0].set_title('Average Price by Vehicle Year')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Count of listings by year
axes[1].bar(year_group['year'], year_group['count'], color='steelblue', alpha=0.7)
axes[1].set_xlabel('Vehicle Year')
axes[1].set_ylabel('Number of Listings')
axes[1].set_title('Listing Count by Vehicle Year')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("price_by_year.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Price by year plotted")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Price Analysis by Vehicle Make", fontsize=14, fontweight='bold')

make_group = preds_all.groupby('make').agg(
    actual_price=('price', 'mean'),
    predicted_price=('predicted_price', 'mean'),
    count=('price', 'count')
).reset_index()

# Filter to makes with at least 2 listings
make_group = make_group[make_group['count'] >= 2]
top10_makes = make_group.nlargest(10, 'actual_price')

x = range(len(top10_makes))
width = 0.35

# Side by side bars
axes[0].bar([i - width/2 for i in x], top10_makes['actual_price'],
            width, label='Actual', color='steelblue', alpha=0.8)
axes[0].bar([i + width/2 for i in x], top10_makes['predicted_price'],
            width, label='Predicted', color='coral', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(top10_makes['make'], rotation=45, ha='right')
axes[0].set_ylabel('Average Price ($)')
axes[0].set_title('Top 10 Makes by Avg Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Listing count by make
top10_count = make_group.nlargest(10, 'count')
axes[1].barh(top10_count['make'], top10_count['count'], color='steelblue', alpha=0.8)
axes[1].set_xlabel('Number of Listings')
axes[1].set_title('Top 10 Makes by Listing Count')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig("price_by_make.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Price by make plotted")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Price vs Mileage Analysis", fontsize=14, fontweight='bold')

mileage_df = preds_all[preds_all['mileage'].between(0, 300_000)].copy()

# Scatter price vs mileage
axes[0].scatter(mileage_df['mileage'], mileage_df['price'],
                alpha=0.5, color='steelblue', edgecolors='k',
                linewidths=0.3, label='Actual')
axes[0].scatter(mileage_df['mileage'], mileage_df['predicted_price'],
                alpha=0.3, color='coral', edgecolors='k',
                linewidths=0.3, label='Predicted')
axes[0].set_xlabel('Mileage')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Price vs Mileage')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Average price by mileage bin
mileage_df['mileage_bin'] = pd.cut(
    mileage_df['mileage'],
    bins=[0, 50_000, 100_000, float('inf')],
    labels=['Low\n(0-50k)', 'Medium\n(50k-100k)', 'High\n(100k+)']
)
bin_group = mileage_df.groupby('mileage_bin', observed=True).agg(
    actual_price=('price', 'mean'),
    predicted_price=('predicted_price', 'mean')
).reset_index()

x = range(len(bin_group))
width = 0.35
axes[1].bar([i - width/2 for i in x], bin_group['actual_price'],
            width, label='Actual', color='steelblue', alpha=0.8)
axes[1].bar([i + width/2 for i in x], bin_group['predicted_price'],
            width, label='Predicted', color='coral', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(bin_group['mileage_bin'])
axes[1].set_ylabel('Average Price ($)')
axes[1].set_title('Avg Price by Mileage Bin')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("price_by_mileage.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Price by mileage plotted")